# Mistral-7B Finance QLoRA — Colab training (zero-manual-edit)

**Run All** works with no cell editing — you only add `HF_TOKEN` (and optional `WANDB_API_KEY`) in Colab **Secrets**. Repo URL, Hub model id, and Drive paths have working defaults in the config cell and are env/Secrets-overridable.

The expensive A100 session is spent ONLY on training. Cheap steps (install, checks, data prep) run on a free T4/CPU and cache the processed dataset to Google Drive; the A100 just restores it.

## Free (T4/CPU) vs paid (A100) split

Run the cheap steps on a **free T4 or CPU** first (they cache the dataset to Drive), then switch to a **paid A100** only for training.

| Cell | Runtime | Purpose |
|---|---|---|
| 1 GPU sanity, 1b config, 2 clone, 3 install, 3b CUDA check | T4 / CPU | setup; does NOT abort on non-Ampere/CPU |
| 3c 4-bit pre-flight | any GPU (T4 ok) | catch a bitsandbytes/CUDA/peft mismatch in seconds (skipped on CPU) |
| 4 secrets, 5 mount Drive, 6 env, 7 data prep | T4 / CPU | prep + cache to Drive |
| 7b train smoke (`RUN_TRAIN_SMOKE=1`) | any GPU | prove the real train loop on a free T4 |
| **8 train, 9 push** | **A100 (paid)** | the ONLY paid step |

**HF_TOKEN is needed even in the free prep session** — the chat template is applied with the gated Mistral tokenizer.

**Recommended order:** run cells 1–7 on a free T4 (dataset cache lands on Drive) → switch runtime to A100 → Run All again (data prep is now a Drive cache hit, no re-download) → cells 8–9 train and push.

**FA2 on a T4:** the Flash-Attention-2 guard only ENFORCES FA2 on Ampere (SM ≥ 8.0). On a T4 (SM 7.5) it degrades to SDPA and does **not** abort, so the prep/validation cells (and the optional train smoke) run cleanly there.

In [ ]:
# 1) GPU sanity. Training (the PAID step) wants an Ampere A100 (SM>=8.0) for
#    FA2 + bf16. The cheap prep/validation cells run fine on a free T4 or CPU,
#    so this cell only REPORTS -- it does NOT abort on non-Ampere or CPU.
import torch
if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability()
    print('GPU:', torch.cuda.get_device_name(0), '| compute capability:', cc)
    if cc[0] >= 8:
        print('Ampere+ detected -> FA2 + bf16 training supported (good for the paid train cell).')
    else:
        print('Non-Ampere GPU (e.g. T4): fine for install / data prep / 4-bit pre-flight / train smoke. '
              'Real training needs an A100 -- the FA2 guard will use SDPA here, not FA2.')
else:
    print('No GPU: only install + data-prep cells are meaningful. '
          'Switch to a GPU runtime for the 4-bit pre-flight and for training.')

In [ ]:
# 1b) Project config. Working defaults; each is overridable via env var or
#     Colab Secret. These are NOT secrets (public repo + public model id), so
#     they are safe to commit. A fresh Run All needs NO edits here.
import os
def _cfg(name, default):
    v = os.environ.get(name)
    if v:
        return v
    try:
        from google.colab import userdata
        s = userdata.get(name)
        if s:
            return s
    except Exception:
        pass
    return default
REPO_URL       = _cfg('REPO_URL', 'https://github.com/mohanemg07-web/Domain-specific-LLM-Fine-tuning.git')
HUB_MODEL_ID   = _cfg('HUB_MODEL_ID', 'MohanGen/mistral7b-finance-qlora')
DRIVE_ROOT     = _cfg('DRIVE_ROOT', '/content/drive/MyDrive/llm_qlora')
DRIVE_DATA_TAR = f'{DRIVE_ROOT}/data_cache.tar.gz'
DRIVE_CKPT     = f'{DRIVE_ROOT}/ckpts/mistral7b-finance-qlora'
print('REPO_URL     =', REPO_URL)
print('HUB_MODEL_ID =', HUB_MODEL_ID)
print('DRIVE_ROOT   =', DRIVE_ROOT)
print('DRIVE_DATA_TAR =', DRIVE_DATA_TAR)
print('DRIVE_CKPT   =', DRIVE_CKPT)

In [ ]:
# 2) Get the code (uses REPO_URL from the config cell -- no placeholder edits).
import os
REPO_DIR = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL
%cd $REPO_DIR

In [ ]:
# 3) Install pinned deps WITHOUT touching Colab's CUDA torch.
#    We pin torch to whatever Colab already ships (constraints file) so
#    nothing in requirements.txt can upgrade/downgrade it. torch is NOT in
#    requirements.txt anymore (see comment there).
import torch
with open('/tmp/constraints.txt', 'w') as f:
    f.write(f'torch=={torch.__version__}\n')
!pip install -q -r requirements.txt -c /tmp/constraints.txt
# flash-attn: NO fixed pin. This picks the prebuilt wheel matching Colab's
# torch/CUDA/cpython/abi (falls back to a --no-build-isolation build).
# On a CPU/T4 prep runtime this is a no-op / SDPA; that's expected.
!python scripts/install_flash_attn.py

In [ ]:
# 3b) HARD CHECK: if we're on a GPU runtime, confirm torch is STILL a CUDA
#     build after the installs. A CPU wheel sneaking in would silently train
#     on CPU and waste paid compute. On a genuine CPU runtime there is nothing
#     to protect, so we just note it (prep-only is fine there).
import shutil, subprocess, torch
print('torch:', torch.__version__, '| torch.version.cuda:', torch.version.cuda,
      '| cuda available:', torch.cuda.is_available())
if torch.version.cuda is None:
    gpu_present = bool(shutil.which('nvidia-smi')) and \
        subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
    assert not gpu_present, (
        'A GPU is present but torch is a CPU build! Something reinstalled torch. '
        'Restart the runtime and reinstall WITHOUT upgrading torch before training.')
    print('CPU torch build and no GPU -- OK for prep-only runs; do NOT train here.')
else:
    assert torch.cuda.is_available(), 'CUDA torch present but no GPU visible -- check runtime.'
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 3c) FAST 4-bit NF4 pre-flight (GPU only). Loads a tiny Llama-arch model in
#     4-bit on the GPU and runs ONE forward pass (plus a PEFT wrap), so a
#     bitsandbytes / CUDA / peft mismatch fails in SECONDS -- before the
#     dataset is processed and the paid session is spent. Skipped on CPU.
import torch
if not torch.cuda.is_available():
    print('No GPU -- skipping 4-bit pre-flight (it needs CUDA). Run it on a T4/A100.')
else:
    import bitsandbytes as bnb
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    print('bitsandbytes:', bnb.__version__,
          '| CUDA it is linked against (torch.version.cuda):', torch.version.cuda)
    _pf_id = 'HuggingFaceTB/SmolLM2-135M'  # tiny, fast, same arch family as Mistral
    _bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                  bnb_4bit_use_double_quant=True,
                                  bnb_4bit_compute_dtype=torch.bfloat16)
    _tok = AutoTokenizer.from_pretrained(_pf_id)
    _m = AutoModelForCausalLM.from_pretrained(_pf_id, quantization_config=_bnb_cfg, device_map={'': 0})
    _m = prepare_model_for_kbit_training(_m)  # exercises the peft<->bnb path too
    _m = get_peft_model(_m, LoraConfig(r=8, lora_alpha=16,
                                       target_modules=['q_proj', 'v_proj'], task_type='CAUSAL_LM'))
    _ids = _tok('4-bit pre-flight', return_tensors='pt').to('cuda')
    with torch.no_grad():
        _out = _m(**_ids)
    print('4-bit NF4 forward OK | logits', tuple(_out.logits.shape), '| device', _out.logits.device)
    del _m, _out, _ids; torch.cuda.empty_cache()

In [ ]:
# 4) Secrets. Prefer Colab Secrets (key icon, left bar): add HF_TOKEN and
#    WANDB_API_KEY there. Falls back to a prompt if not set. HF_TOKEN is needed
#    even for prep (gated Mistral tokenizer drives the chat template).
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    try:
        os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    except Exception:
        print('No WANDB_API_KEY secret — W&B logging will be disabled.')
except Exception:
    from getpass import getpass
    os.environ['HF_TOKEN'] = getpass('HF_TOKEN (needs Mistral-7B license access): ')
    os.environ['WANDB_API_KEY'] = getpass('WANDB_API_KEY (blank to skip): ')
from huggingface_hub import login
login(os.environ['HF_TOKEN'])

In [ ]:
# 5) Mount Drive (the durable data cache + checkpoints live here) and ensure
#    the directories exist. Training writes checkpoints straight to DRIVE_CKPT
#    and resumes from it, so a disconnect costs at most save_steps steps.
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(os.path.dirname(DRIVE_CKPT), exist_ok=True)
print('Drive ready. DRIVE_ROOT =', DRIVE_ROOT)

In [ ]:
# 6) Show the runtime guard decisions. On an A100 expect flash_attention_2 /
#    uses_4bit_quant True / training_bf16 True. On a T4 expect sdpa (no abort).
from src.config import describe_environment
print(describe_environment())

In [ ]:
# 7) Data prep -- computed ONCE on free compute, then the A100 just restores.
#    restore (Drive tar -> local data_cache/) -> prepare_dataset (a no-op cache
#    hit if restored, else download+process) -> snapshot (persist back to Drive).
#    data_cache_dir stays LOCAL ('data_cache') for fast Arrow I/O.
LOCAL_DATA = 'data_cache'
!python scripts/drive_cache.py restore --drive "$DRIVE_DATA_TAR" --local "$LOCAL_DATA"
from src.config import load_settings
from src.data.prepare import prepare_dataset
settings = load_settings()  # smoke_test=False -> real Mistral-7B + 50k subsample; cache_dir='data_cache'
manifest = prepare_dataset(settings)
print(manifest)  # 'cache_hit': True on a fresh runtime that restored the Drive tarball
!python scripts/drive_cache.py snapshot --local "$LOCAL_DATA" --drive "$DRIVE_DATA_TAR"

In [ ]:
# 7b) OPTIONAL free-tier TRAINING SMOKE (default OFF). Set RUN_TRAIN_SMOKE=1 to
#     run 5 REAL train() steps on a tiny real-data subset and checkpoint to a
#     throwaway Drive dir -- proving the SFTTrainer wiring + Drive checkpointing
#     on a free T4 BEFORE you pay for the A100. A normal Run All SKIPS this.
import os
if os.environ.get('RUN_TRAIN_SMOKE', '').strip().lower() in ('1', 'true', 'yes', 'on'):
    from dataclasses import replace
    from src.data.prepare import load_splits
    from src.train.sft import train
    _train_ds, _ = load_splits(settings)
    _subset = _train_ds.select(range(min(len(_train_ds), 50)))
    _smoke_dir = f'{DRIVE_ROOT}/train_smoke_ckpt'  # throwaway; not the real DRIVE_CKPT
    _s = replace(settings, max_steps=5, save_steps=5, logging_steps=1,
                 per_device_train_batch_size=1, gradient_accumulation_steps=1,
                 output_dir=_smoke_dir, push_to_hub=False, resume_from_checkpoint=False,
                 report_to='none')
    _m = train(_s, train_dataset=_subset)
    print('TRAIN SMOKE OK (5 real steps):', _m)
else:
    print('RUN_TRAIN_SMOKE not set -- skipping free-tier training smoke (normal Run All).')

In [ ]:
# 8) TRAIN (PAID A100 step). Checkpoints straight to Drive (DRIVE_CKPT) every
#    save_steps; auto-resumes from the latest checkpoint there. Reuses the same
#    LOCAL data_cache so data prep above is a cache hit (no re-download).
#    FA2 enforcement: on an A100 this ABORTS if flash_attention_2 is not active.
#    If flash-attn genuinely won't install and you'd rather proceed on SDPA than
#    burn the session: import os; os.environ['ALLOW_SDPA_FALLBACK'] = '1'
#    (default off; metrics then record attn_implementation_active='sdpa').
from dataclasses import replace
from src.train.sft import train
settings = replace(settings, output_dir=DRIVE_CKPT, resume_from_checkpoint=True,
                   push_to_hub=True, hub_model_id=HUB_MODEL_ID)
metrics = train(settings)
print('REAL metrics from this A100 run:', metrics)  # peak_vram_gb etc. are measured

In [ ]:
# 9) Push the final adapter to the Hub (repo id = HUB_MODEL_ID from the config cell).
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(settings.hub_model_id, exist_ok=True)
api.upload_folder(folder_path=metrics['adapter_dir'], repo_id=settings.hub_model_id)
print('Pushed adapter to', settings.hub_model_id)